In [2]:
import numpy as np
import networkx as nx
import pandas as pd
from itertools import combinations as combs
from matplotlib import pyplot as plt
np.set_printoptions(threshold=20)
import os.path as osp

## Plot the results where attractors are weighted by transcriptional data

### This figure relies on the results generated by the `attractor_weighted_irreversibility.py` script

In [3]:


#hamming_summaries_default = pd.read_pickle('results/crp/hamming_summaries.pkl')
#hamming_summaries_average = pd.read_pickle('results/crp/hamming_summaries_average.pkl')
hamming_summaries_median = pd.read_pickle('results/crp/hamming_summaries_median.pkl')
#hamming_summaries_zero = pd.read_pickle('results/crp/hamming_summaries_zero.pkl')
#len(boltz_summaries.keys())

#hamm_avg = pd.concat(hamming_summaries_average,axis=1).T.groupby(level=2).mean().T
#sel_hamm_avg = hamm_avg[(hamm_avg.irr>1e-3)].sort_values('irr')
#hamm_default = pd.concat(hamming_summaries_default,axis=1).T.groupby(level=2).mean().T
#sel_hamm_default = hamm_default[(hamm_default.irr>1e-3)].sort_values('irr')
hamm_median = pd.concat(hamming_summaries_median,axis=1).T.groupby(level=2).mean().T
sel_hamm_median = hamm_median[(hamm_median.irr>1e-3)].sort_values('irr')
#hamm_zero = pd.concat(hamming_summaries_zero,axis=1).T.groupby(level=2).mean().T
#sel_hamm_zero = hamm_zero[(hamm_zero.irr>1e-3)].sort_values('irr')

hamming_median_std = pd.concat(hamming_summaries_median,axis=1).T.groupby(level=2).std().T

,default,average,median,zero
metJ,0.037859,0.060176,NaN,NaN
fur,0.038923,0.011558,NaN,NaN
xylR,0.044118,0.011437,0.064202,NaN
cadC,0.045977,0.050458,0.061732,0.125353
nikR,0.056795,0.038063,0.061299,0.002669
gutM,0.057414,0.032447,0.067549,NaN
soxS,0.057478,0.062203,0.074411,0.014162
ptsG,0.060521,0.095098,0.010221,0.026447
galR,0.062031,0.081840,0.036644,0.434457
csgD,0.063239,0.012086,0.097503,NaN


In [5]:
#boltz_summaries_default = pd.read_pickle('results/crp/boltz_summaries.pkl')
#boltz_summaries_average = pd.read_pickle('results/crp/boltz_summaries_average.pkl')
boltz_summaries_median = pd.read_pickle('results/crp/boltz_summaries_median.pkl')
#boltz_summaries_zero = pd.read_pickle('results/crp/boltz_summaries_zero.pkl')
#len(boltz_summaries.keys())

#boltz_avg = pd.concat(boltz_summaries_average,axis=1).T.groupby(level=2).mean().T
#sel_boltz_avg = boltz_avg[(boltz_avg.irr>1e-3)].sort_values('irr')
#boltz_default = pd.concat(boltz_summaries_default,axis=1).T.groupby(level=2).mean().T
#sel_boltz_default = boltz_default[(boltz_default.irr>1e-3)].sort_values('irr')
boltz_median = pd.concat(boltz_summaries_median,axis=1).T.groupby(level=2).mean().T
sel_boltz_median = boltz_median[(boltz_median.irr>1e-3)].sort_values('irr')
#boltz_zero = pd.concat(boltz_summaries_zero,axis=1).T.groupby(level=2).mean().T
#sel_boltz_zero = boltz_zero[(boltz_zero.irr>1e-3)].sort_values('irr')
boltz_median_std = pd.concat(boltz_summaries_median,axis=1).T.groupby(level=2).std().T


,default,average,median,zero
nac,0.054506,0.014948,0.005070,0.002167
dpiA,0.058518,0.066285,0.002892,0.067644
dcuR,0.059999,0.027064,0.003679,0.022896
soxR,0.060419,0.034956,0.028133,0.477005
ptsG,0.069275,0.021766,0.003820,0.044333
marA,0.070681,0.006271,NaN,0.342516
arcA,0.074129,0.070033,0.035341,0.575355
narL,0.074129,0.070033,0.035341,0.575355
marR,0.081143,0.013787,0.057221,0.087381
puuR,0.086170,0.078938,0.022136,0.427377


In [ ]:
G_rs2 = nx.read_gml('networks/rs2.gml')
## loop over nodes and attempt to categorize:
from collections import defaultdict
nd_crpinfo_dd = defaultdict(dict)
for nd in G_rs2.nodes():
    if nd in ['dnaA','argP','phoB','cra','pdeL','cusR']:
        nd_crpinfo_dd[nd]['crp_regulation']='none'
        nd_crpinfo_dd[nd]['crp_reg_sign']='0'
        nd_crpinfo_dd[nd]['crp_reg_len']=88
        if nd in G_rs2.predecessors(nd):
            if G_rs2.edges[nd,nd]['weight']>0:
                nd_crpinfo_dd[nd]['self_regulation']='+'
            else:
                nd_crpinfo_dd[nd]['self_regulation']='-'
        else:
            nd_crpinfo_dd[nd]['self_regulation']='0'
        continue    
    elif 'crp' in G_rs2.predecessors(nd):
        preds = [elt for elt in G_rs2.predecessors(nd)]
        nd_crpinfo_dd[nd]['crp_reg_len']=1
        if len(preds)==1 or (len(preds)==2 and nd in preds):
            nd_crpinfo_dd[nd]['crp_regulation']='exclusive direct'
        else:
            nd_crpinfo_dd[nd]['crp_regulation']='nonexclusive direct'
        if G_rs2.edges[('crp',nd)]['weight'] > 0:
            nd_crpinfo_dd[nd]['crp_reg_sign']='+'
        else:
            nd_crpinfo_dd[nd]['crp_reg_sign']='-'
    else:
        preds = [elt for elt in G_rs2.predecessors(nd)]
        nd_crpinfo_dd[nd]['crp_regulation']='indirect'
        #find all simple paths to crp
        len_sgn_l = []
        for pp in nx.all_simple_paths(G_rs2,'crp',nd):
            #pp[0] is 'crp'
            #pp[-1] is nd
            ## calculate length 
            ll = len(pp)-1
            ## calculate sign
            ss = np.prod([G_rs2.edges[edg]['weight'] for edg in zip(pp[:-1],pp[1:])])
            len_sgn_l.append((ll,ss))
        ll_l,ss_l = zip(*len_sgn_l)
        len_sgn_l = sorted(len_sgn_l,key = lambda xx: xx[0])
        l_min, s_min = len_sgn_l[0]
        nd_crpinfo_dd[nd]['crp_reg_len']=l_min
        if np.all(np.asarray(ss_l)>0):
            nd_crpinfo_dd[nd]['crp_reg_sign']='+'
        elif np.all(np.asarray(ss_l)<0):
            nd_crpinfo_dd[nd]['crp_reg_sign']='-'
        else:
            if s_min>0:
                nd_crpinfo_dd[nd]['crp_reg_sign']='+'
            else:
                nd_crpinfo_dd[nd]['crp_reg_sign']='-'
            for ll,ss in len_sgn_l[1:]:
                if ll>l_min+1:
                    break
                if ss != s_min:
                    nd_crpinfo_dd[nd]['crp_reg_sign']='both'
                    break
    # calculate the self regulations
    if nd in preds:
        if G_rs2.edges[nd,nd]['weight']>0:
            nd_crpinfo_dd[nd]['self_regulation']='+'
        else:
            nd_crpinfo_dd[nd]['self_regulation']='-'
    else:
        nd_crpinfo_dd[nd]['self_regulation']='0'
        
    
nd_crpinfo_df = pd.DataFrame(dict(nd_crpinfo_dd)).T

In [ ]:
plt.rcParams['svg.fonttype']='none'
fc_d = {'+':'#EDB120','-':'C0','both':'C2'}
ec_d = {'+':'#EDB120','-':'C0','0':'C7'}
al_dec = 0.15
marker_size = 64
LW = 2

## need to break up by information:
## the distance is related to the alpha
## the edge color is related to the SELF regulation
## the face color is related to the crp regulation (to maintain consistency with Fig. 7)

hamming_median = pd.concat(hamming_summaries_median,axis=1).T.groupby(level=2).mean().T
non_reg = nd_crpinfo_df[nd_crpinfo_df.crp_regulation=='none']
fig,ax_l = plt.subplots(1,2,figsize=(6.5,4),sharey=True)

for ii,ax in enumerate(ax_l):
    if ii==0:
        xvals = 1-hamming_median.unch
        yvals = hamming_median.irr/(1-hamming_median.unch)
        xerr = hamming_median.unch/np.sqrt(15)
        bcyerr = hamming_median.irr/(1-hamming_median.unch) * np.sqrt(((hamming_median_std.irr/hamming_median.irr)**2 + (hamming_median_std.unch/(1-hamming_median.unch))**2))
        yerr = bcyerr/np.sqrt(15)
        ax.set_title('Hamming',size=10)
        ax.set_ylabel('Conditional irreversible response probability',size=8)
        ax.set_ylim(-0.02,1.02)
        plt.setp(ax.get_yticklabels(),size=8)
    else:
        xvals = 1-boltz_median.unch
        yvals = boltz_median.irr/(1-boltz_median.unch)
        xerr = boltz_median_std.unch/np.sqrt(15)
        bcyerr = boltz_median.irr/(1-boltz_median.unch) * np.sqrt(((boltz_median_std.irr/boltz_median.irr)**2 + (boltz_median_std.unch/(1-boltz_median.unch))**2))
        yerr = bcyerr/np.sqrt(15)
        ax.set_title('Likelihood',size=10)

    #non_eb = ax.errorbar(xvals.loc[non_reg.index],yvals.loc[non_reg.index],xerr=xerr.loc[non_reg.index],yerr=yerr.loc[non_reg.index],
    #                     alpha=0.5, markeredgecolor='C7', markerfacecolor='C7', ecolor='C7',marker='o',label=(88,'none','NA'))
    non_eb = ax.scatter(xvals.loc[non_reg.index],yvals.loc[non_reg.index],alpha=0.4, edgecolor='none', facecolor='C7', marker='o', label=(88,'none','NA'))
    
    for ll in [0,1,2,3,4]:
        if ll==0:
            cLen = (nd_crpinfo_df.crp_reg_len==1)&(nd_crpinfo_df.crp_regulation=='exclusive direct')&(nd_crpinfo_df.index!='crp')
        elif ll==1:
            cLen = (nd_crpinfo_df.crp_reg_len==ll)&(nd_crpinfo_df.crp_regulation=='nonexclusive direct')&(nd_crpinfo_df.index!='crp')
        else:
            cLen = (nd_crpinfo_df.crp_reg_len==ll)
        al = 1-(ll)*al_dec
        for crs in ['+','-','both']:
            fc = fc_d[crs]
            cCRS = (nd_crpinfo_df.crp_reg_sign==crs)
            for srs in ['+','-','0']:
                ec = ec_d[srs]
                cSRS = (nd_crpinfo_df.self_regulation==srs)
                sel_inds = nd_crpinfo_df[cLen&cCRS&cSRS]
                # hndl = ax.errorbar(xvals.loc[sel_inds.index],yvals.loc[sel_inds.index],xerr=xerr.loc[sel_inds.index],yerr=yerr.loc[sel_inds.index],
                #                    alpha=al, markeredgecolor=ec, markerfacecolor=fc, ecolor=ec,marker='o',ls='',label=(ll,crs,srs))
                hndl = ax.scatter(xvals.loc[sel_inds.index],yvals.loc[sel_inds.index], s=marker_size, linewidth=LW, alpha=al, 
                                  edgecolor=ec, facecolor=fc,marker='o',label=(ll,crs,srs))
    ax.set_xlabel('Probability of changing',size=8)
    ax.set_xlim(-0.01,0.4)
    plt.setp(ax.get_xticklabels(),size=8)
    
    
#leg = fig.legend(ncol=4, frameon=False)
fig.savefig('figs/figs6_attractor_weighted_irgs.svg')